# Automatic Differentiation with DifferentiationInterface

DifferentiationInterface.jl offers one set of functions (`derivative`, `gradient`, `jacobian`, `hessian`, ...) over many *backends*. Forward-mode backends (ForwardDiff) cost about one pass per input; reverse-mode backends (Enzyme, Mooncake) give a whole gradient in about one backward pass. This notebook runs the same code on each backend and times the difference.

In [ ]:
Pollis.packages("DifferentiationInterface", "ForwardDiff", "FiniteDiff", "Plots")

In [ ]:
using DifferentiationInterface
import ForwardDiff, FiniteDiff
for pkg in (:Enzyme, :Mooncake)   # optional backends
    try
        @eval import $pkg
    catch
        @warn "$pkg is not installed; its results will be skipped"
    end
end

## One function, five backends

In [ ]:
f(x) = sum(abs2, x) / 2 + prod(cos, x)
x = [0.5, 1.0, 1.5]
backends = ["ForwardDiff" => AutoForwardDiff(), "Enzyme" => AutoEnzyme(), "Mooncake" => AutoMooncake(; config=nothing), "FiniteDiff" => AutoFiniteDiff()]
for (name, b) in backends
    try
        println(rpad(name, 12), gradient(f, b, x))
    catch e
        println(rpad(name, 12), "failed: ", typeof(e))
    end
end

## Preparation

For repeated calls at different points, `prepare_gradient` does the one-off work (tracing, compiling rules, allocating buffers) once. Pass the preparation object to later calls.

In [ ]:
b = AutoForwardDiff()
prep = prepare_gradient(f, b, x)
g = similar(x)
gradient!(f, g, prep, b, [1.0, 2.0, 3.0])   # in place, no allocation of the result
g

## Forward vs reverse mode as the number of inputs grows

Forward mode needs work proportional to the number of inputs $n$; reverse mode does not.

In [ ]:
using Plots
ns = [2, 10, 50, 200, 1000]
function time_gradient(b, n)
    x = rand(n); prep = prepare_gradient(f, b, x); gradient(f, prep, b, x)
    return 1e6 * (@elapsed for _ in 1:20; gradient(f, prep, b, x); end) / 20
end
p = plot(xscale=:log10, yscale=:log10, xlabel="n inputs", ylabel="microseconds per gradient", legend=:topleft)
for (name, b) in backends[1:3]
    try
        plot!(ns, [time_gradient(b, n) for n in ns], m=:circle, lw=2, label=name)
    catch
        @warn "skipping $name"
    end
end
p

## Exercises

1. Write `f` with a loop that updates a preallocated array (mutation). Which backends still work?
2. Compute a Hessian with `hessian(f, SecondOrder(AutoForwardDiff(), AutoEnzyme()), x)` (forward over reverse).
3. Read the Automatic Differentiation wiki page and redo its worked example with ForwardDiff.